In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CSV_PATH = 'path/to/Kulingtang_metadata.csv'

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
df = df[df['label'] != 'N']
df.head()

,video_name,video_root_folder,audio_name,audio_root_folder,video_segment_name,video_segment_path,label
4,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part5.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,2
5,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part6.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,3
6,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part7.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,4
7,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part8.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,5
11,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part12.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,8


Getting all the video segments, extracting audio, converting to features and saving to csv

In [ ]:
!pip install moviepy librosa matplotlib numpy -q

In [ ]:
from moviepy.editor import VideoFileClip

# Load the video file
def extract_audio(video_path):
  video = VideoFileClip(video_path)

  # Extract and save the audio
  audio = video.audio
  audio.write_audiofile("audio.wav")

  if event.key is 'enter':



In [ ]:
import librosa #
import pandas as pd
import numpy as np

#function to extract features from audion files
def extract_features(audio_path):

    # load the audio file
    y,sr = librosa.load(audio_path,mono=True) #load the audio file
    # extract features
    rmse = librosa.feature.rms(y=y)[0] #compute root-mean-square (RMS) value for each frame
    spec_cent = librosa.feature.spectral_centroid(y=y,sr=sr) #calculate spectral_centroid
    spec_bw = librosa.feature.spectral_bandwidth(y=y,sr=sr) #calculate spectral_bandwith
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr) #calculate spectral_rolloff
    zcr = librosa.feature.zero_crossing_rate(y) #calculate zero crossing rate
    mfcc = librosa.feature.mfcc(y=y,sr=sr) #Mel frequency ceptral coefficients (mfcc)


    audio_dict={
        'RMSE':rmse.mean(),
        'SPECTRAL_CENTROID':spec_cent.mean(),
        'SPECTRAL_BANDWIDTH':spec_bw.mean(),
        'ROLLOFF':rolloff.mean(),
        'ZERO_CROSSING_RATE':zcr.mean()
    }

    #add the mfcc values
    for index,mfcc_set in enumerate(mfcc):
        feature_name=f"MFCC_FEATURE_{index}"
        audio_dict[feature_name]=np.mean(mfcc_set)

    return audio_dict

In [ ]:
from tqdm.notebook import tqdm

# Extract audio features of each record
audio_features = []
audio_file_path = "audio.wav"
for file_path in tqdm(df['video_segment_path'], desc="Processing Files"):
  extract_audio(file_path)
  features = extract_features(audio_file_path)
  audio_features.append(features)

features_df = pd.DataFrame(audio_features)
df_new = pd.concat([df["video_segment_name"].reset_index(drop=True), features_df], axis=1)
df_new.head()

In [ ]:
df_label = df["label"]
df_label = df_label.reset_index(drop=True)
print(df_label)

0       2
1       3
2       4
3       5
4       8
       ..
1076    1
1077    2
1078    3
1079    6
1080    7
Name: label, Length: 1081, dtype: object


In [ ]:
# Concatenate the DataFrames
df_main = pd.concat([df_new, df_label], axis=1)

df_main.head()

,video_segment_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,label
0,IMG_0037_part5.mov,0.110914,636.822926,730.140212,704.956055,0.042457,-388.798676,127.769531,30.357565,0.964474,...,-9.094934,2.113101,9.804667,13.554675,10.162496,5.915237,7.076754,9.991291,4.527101,2
1,IMG_0037_part6.mov,0.105051,802.982334,848.353061,964.883256,0.048331,-370.882019,147.863907,10.282371,-6.605198,...,-0.436702,17.252117,17.207047,9.353371,2.422301,2.440640,4.783227,-7.288693,-19.787867,3
2,IMG_0037_part7.mov,0.117851,824.580504,817.884068,992.065430,0.050767,-356.948273,154.975510,5.095802,-17.982309,...,11.097884,14.794453,7.392744,-1.549907,-4.859144,-3.377399,-8.645940,-23.538467,-19.215620,4
3,IMG_0037_part8.mov,0.070839,1288.281722,1206.854531,1982.711088,0.076848,-268.320526,161.854477,-57.438450,-28.290646,...,14.017269,-4.714746,-6.482289,1.976948,-0.569431,-14.925186,-21.212166,-12.418089,-4.545397,5
4,IMG_0037_part12.mov,0.098182,1416.618909,981.970283,1715.254211,0.091492,-315.661011,115.525948,-67.717117,-48.924000,...,-9.691248,7.581002,0.971636,0.144252,6.229625,-5.574753,-19.128342,-13.515326,9.690474,8


In [ ]:
# Check for duplicate values
df_main.duplicated().sum()

np.int64(0)

In [ ]:
# Check for null values
df_main.isnull().sum()

,0
video_segment_name,0
RMSE,0
SPECTRAL_CENTROID,0
SPECTRAL_BANDWIDTH,0
ROLLOFF,0
ZERO_CROSSING_RATE,0
MFCC_FEATURE_0,0
MFCC_FEATURE_1,0
MFCC_FEATURE_2,0
MFCC_FEATURE_3,0


In [ ]:
# Save the DataFrame to a CSV file
df_main.to_csv('path/to/featurizedaudio.csv', index=False)